In [1]:
print("Hello world")

Hello world


In [2]:
import pandas as pd 

train_df = pd.read_csv(r"predictions\train_df.csv")
train_df.label = train_df.label.fillna("None")
train_df.head()

test_df = pd.read_csv(r"predictions\test_df.csv")
test_df.label = test_df.label.fillna("None")
test_df.head()

,case_name,sent_text,label,hdr_title,hdr_match,hdr_group,y_labelled
0,ECLI:NL:GHAMS:2015:2960.txt,De kantonrechter heeft in het bestreden vonnis...,None,Feiten,1,Feiten,0
1,ECLI:NL:GHAMS:2015:2960.txt,Deze feiten zijn in hoger beroep niet in gesch...,None,Feiten,1,Feiten,0
2,ECLI:NL:GHAMS:2015:2960.txt,Op [datum] heeft [geïntimeerde] een bedrag van...,materiele feiten,Beoordeling,1,Beoordeling,1
3,ECLI:NL:GHAMS:2015:2960.txt,Op [datum] heeft [appellante] een schriftelijk...,materiele feiten,Beoordeling,1,Beoordeling,1
4,ECLI:NL:GHAMS:2015:2960.txt,Deze verklaring houdt onder meer in: “Dit bedr...,None,Beoordeling,1,Beoordeling,0


In [3]:
# ============================================================
# Logistic Regression baselines for:
# 1. 5-way classification
# 2. Stage 1 binary gatekeeper
# 3. Stage 2 4-way classification
# 4. OvR binary models
#
# Stores all outputs in lr_output_df
# ============================================================

import os
import re
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression


# ------------------------------------------------------------
# 0. Data setup
# ------------------------------------------------------------

# Training dataframe
lr_train_df = train_df.copy()

# Dataframe where predictions/probabilities will be stored
# Usually this is your test dataframe with transformer outputs already attached
lr_output_df = test_df.copy()

TEXT_COL = "sent_text"
GOLD_COL = "label"

lr_train_df[TEXT_COL] = lr_train_df[TEXT_COL].fillna("").astype(str)
lr_output_df[TEXT_COL] = lr_output_df[TEXT_COL].fillna("").astype(str)


LABELS_5WAY = [
    "None",
    "beoordeling",
    "beslissing",
    "materiele feiten",
    "proceshandelingen"
]

LEGAL_LABELS = [
    "beoordeling",
    "beslissing",
    "materiele feiten",
    "proceshandelingen"
]


# ------------------------------------------------------------
# 1. Helper functions
# ------------------------------------------------------------

def clean_col_name(x):
    x = str(x).strip().lower()
    x = re.sub(r"\s+", "_", x)
    x = re.sub(r"[^a-zA-Z0-9_]+", "_", x)
    x = re.sub(r"_+", "_", x).strip("_")
    return x


def make_lr_pipeline(class_weight="balanced"):
    """
    TF-IDF + Logistic Regression.
    
    Word n-grams capture legal phrases.
    Character n-grams help with Dutch morphology and spelling variation.
    """
    features = FeatureUnion([
        ("word_tfidf", TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True
        )),
        ("char_tfidf", TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True
        ))
    ])

    clf = LogisticRegression(
        class_weight=class_weight,
        solver="lbfgs",
        max_iter=5000,
        n_jobs=-1,
        random_state=42
    )

    return Pipeline([
        ("features", features),
        ("clf", clf)
    ])


def add_proba_columns(df, prefix, model, proba):
    """
    Add probability columns using model.classes_ order.
    """
    for i, lab in enumerate(model.classes_):
        clean_lab = clean_col_name(lab)
        df[f"{prefix}_p_{clean_lab}"] = proba[:, i]
    return df


# ------------------------------------------------------------
# 2. Logistic Regression 5-way model
# ------------------------------------------------------------

print("Training Logistic Regression 5-way model...")

lr_5way = make_lr_pipeline(class_weight="balanced")

lr_5way.fit(
    lr_train_df[TEXT_COL],
    lr_train_df[GOLD_COL]
)

lr_output_df["lr_5way_pred_label"] = lr_5way.predict(
    lr_output_df[TEXT_COL]
)

lr_5way_proba = lr_5way.predict_proba(
    lr_output_df[TEXT_COL]
)

lr_output_df = add_proba_columns(
    df=lr_output_df,
    prefix="lr_5way",
    model=lr_5way,
    proba=lr_5way_proba
)


# ------------------------------------------------------------
# 3. Logistic Regression Stage 1 gatekeeper
# Binary: None vs labelled
# ------------------------------------------------------------

print("Training Logistic Regression Stage 1 gatekeeper...")

lr_stage1_train = lr_train_df.copy()

lr_stage1_train["y_stage1"] = np.where(
    lr_stage1_train[GOLD_COL] == "None",
    0,
    1
)

lr_stage1 = make_lr_pipeline(class_weight="balanced")

lr_stage1.fit(
    lr_stage1_train[TEXT_COL],
    lr_stage1_train["y_stage1"]
)

lr_output_df["lr_stage1_pred_idx"] = lr_stage1.predict(
    lr_output_df[TEXT_COL]
)

lr_output_df["lr_stage1_pred_label"] = np.where(
    lr_output_df["lr_stage1_pred_idx"] == 0,
    "not_labelled",
    "labelled"
)

lr_stage1_proba = lr_stage1.predict_proba(
    lr_output_df[TEXT_COL]
)

# Since classes are [0, 1], this stores:
# lr_stage1_p0 = probability of None / not labelled
# lr_stage1_p1 = probability of labelled
class_to_index = {cls: i for i, cls in enumerate(lr_stage1.classes_)}

lr_output_df["lr_stage1_p0"] = lr_stage1_proba[:, class_to_index[0]]
lr_output_df["lr_stage1_p1"] = lr_stage1_proba[:, class_to_index[1]]

# Convenient aliases
lr_output_df["lr_p_none"] = lr_output_df["lr_stage1_p0"]
lr_output_df["lr_p_labelled"] = lr_output_df["lr_stage1_p1"]


# ------------------------------------------------------------
# 4. Logistic Regression Stage 2 4-way model
# Train only on legal-labelled sentences
# ------------------------------------------------------------

print("Training Logistic Regression Stage 2 4-way model...")

lr_stage2_train = lr_train_df[
    lr_train_df[GOLD_COL].isin(LEGAL_LABELS)
].copy()

lr_stage2 = make_lr_pipeline(class_weight="balanced")

lr_stage2.fit(
    lr_stage2_train[TEXT_COL],
    lr_stage2_train[GOLD_COL]
)

# Predict on all rows so you can later test different routing rules
lr_output_df["lr_stage2_pred_label"] = lr_stage2.predict(
    lr_output_df[TEXT_COL]
)

lr_stage2_proba = lr_stage2.predict_proba(
    lr_output_df[TEXT_COL]
)

lr_output_df = add_proba_columns(
    df=lr_output_df,
    prefix="lr_stage2",
    model=lr_stage2,
    proba=lr_stage2_proba
)


# ------------------------------------------------------------
# 5. Logistic Regression OvR models
# One binary model per legal label
# Train only on legal-labelled sentences
# ------------------------------------------------------------

print("Training Logistic Regression OvR models...")

lr_ovr_train = lr_train_df[
    lr_train_df[GOLD_COL].isin(LEGAL_LABELS)
].copy()

lr_ovr_models = {}

for target_label in LEGAL_LABELS:
    
    print(f"  Training OvR model for: {target_label}")
    
    clean_label = clean_col_name(target_label)
    
    y_binary = np.where(
        lr_ovr_train[GOLD_COL] == target_label,
        1,
        0
    )
    
    model = make_lr_pipeline(class_weight="balanced")
    
    model.fit(
        lr_ovr_train[TEXT_COL],
        y_binary
    )
    
    lr_ovr_models[target_label] = model
    
    pred = model.predict(lr_output_df[TEXT_COL])
    proba = model.predict_proba(lr_output_df[TEXT_COL])
    
    class_to_index = {cls: i for i, cls in enumerate(model.classes_)}
    
    lr_output_df[f"lr_{clean_label}_pred"] = pred
    lr_output_df[f"lr_{clean_label}_p_not"] = proba[:, class_to_index[0]]
    lr_output_df[f"lr_{clean_label}_p_pos"] = proba[:, class_to_index[1]]


# ------------------------------------------------------------
# 6. Optional combined Logistic Regression pipeline predictions
# ------------------------------------------------------------

# Strict Stage 1 + Stage 2 pipeline
lr_output_df["lr_pipeline_stage1_stage2_pred"] = np.where(
    lr_output_df["lr_stage1_pred_idx"] == 0,
    "None",
    lr_output_df["lr_stage2_pred_label"]
)

# OvR winner among positive probabilities
lr_ovr_pos_cols = [
    "lr_beoordeling_p_pos",
    "lr_beslissing_p_pos",
    "lr_materiele_feiten_p_pos",
    "lr_proceshandelingen_p_pos"
]

ovr_col_to_label = {
    "lr_beoordeling_p_pos": "beoordeling",
    "lr_beslissing_p_pos": "beslissing",
    "lr_materiele_feiten_p_pos": "materiele feiten",
    "lr_proceshandelingen_p_pos": "proceshandelingen"
}

best_lr_ovr_col = lr_output_df[lr_ovr_pos_cols].idxmax(axis=1)

lr_output_df["lr_ovr_best_label"] = best_lr_ovr_col.map(ovr_col_to_label)
lr_output_df["lr_ovr_best_score"] = lr_output_df[lr_ovr_pos_cols].max(axis=1)

# Strict Stage 1 + OvR pipeline
lr_output_df["lr_pipeline_stage1_ovr_pred"] = np.where(
    lr_output_df["lr_stage1_pred_idx"] == 0,
    "None",
    lr_output_df["lr_ovr_best_label"]
)


# ------------------------------------------------------------
# 7. Save output
# ------------------------------------------------------------

output_path = "predictions/lr_all_model_outputs.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

lr_output_df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("\nDone.")
print("Final shape:", lr_output_df.shape)
print("Saved to:", output_path)

lr_output_df.head()

Training Logistic Regression 5-way model...
Training Logistic Regression Stage 1 gatekeeper...
Training Logistic Regression Stage 2 4-way model...
Training Logistic Regression OvR models...
  Training OvR model for: beoordeling
  Training OvR model for: beslissing
  Training OvR model for: materiele feiten
  Training OvR model for: proceshandelingen

Done.
Final shape: (1502, 40)
Saved to: predictions/lr_all_model_outputs.csv


,case_name,sent_text,label,hdr_title,hdr_match,hdr_group,y_labelled,lr_5way_pred_label,lr_5way_p_none,lr_5way_p_beoordeling,...,lr_materiele_feiten_pred,lr_materiele_feiten_p_not,lr_materiele_feiten_p_pos,lr_proceshandelingen_pred,lr_proceshandelingen_p_not,lr_proceshandelingen_p_pos,lr_pipeline_stage1_stage2_pred,lr_ovr_best_label,lr_ovr_best_score,lr_pipeline_stage1_ovr_pred
0,ECLI:NL:GHAMS:2015:2960.txt,De kantonrechter heeft in het bestreden vonnis...,None,Feiten,1,Feiten,0,None,0.315732,0.198464,...,0,0.575514,0.424486,0,0.604362,0.395638,None,beoordeling,0.506569,None
1,ECLI:NL:GHAMS:2015:2960.txt,Deze feiten zijn in hoger beroep niet in gesch...,None,Feiten,1,Feiten,0,beoordeling,0.259081,0.636357,...,0,0.845994,0.154006,0,0.863053,0.136947,None,beoordeling,0.901524,None
2,ECLI:NL:GHAMS:2015:2960.txt,Op [datum] heeft [geïntimeerde] een bedrag van...,materiele feiten,Beoordeling,1,Beoordeling,1,materiele feiten,0.374714,0.052794,...,1,0.151273,0.848727,0,0.824825,0.175175,None,materiele feiten,0.848727,None
3,ECLI:NL:GHAMS:2015:2960.txt,Op [datum] heeft [appellante] een schriftelijk...,materiele feiten,Beoordeling,1,Beoordeling,1,materiele feiten,0.255660,0.077961,...,1,0.137402,0.862598,0,0.847397,0.152603,materiele feiten,materiele feiten,0.862598,materiele feiten
4,ECLI:NL:GHAMS:2015:2960.txt,Deze verklaring houdt onder meer in: “Dit bedr...,None,Beoordeling,1,Beoordeling,0,None,0.372447,0.339269,...,0,0.649835,0.350165,0,0.791317,0.208683,beoordeling,beoordeling,0.587207,beoordeling


## 5 Way Classification

In [4]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import pandas as pd

# ------------------------------------------------------------
# 5-way classification evaluation
# ------------------------------------------------------------

LABELS_5WAY = [
    "None",
    "beoordeling",
    "beslissing",
    "materiele feiten",
    "proceshandelingen"
]

y_true = lr_output_df["label"]
y_pred = lr_output_df["lr_5way_pred_label"]

# ------------------------------------------------------------
# Accuracy
# ------------------------------------------------------------

acc = accuracy_score(y_true, y_pred)
print("5-way classification accuracy:", round(acc, 4))

# ------------------------------------------------------------
# Classification report
# ------------------------------------------------------------

print("\nClassification Report - 5-way model")
print(
    classification_report(
        y_true,
        y_pred,
        labels=LABELS_5WAY,
        target_names=LABELS_5WAY,
        digits=4,
        zero_division=0
    )
)

# ------------------------------------------------------------
# Confusion matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=LABELS_5WAY
)
print("\nRaw Confusion Matrix (5-way model)")
print(cm)

cm_df = pd.DataFrame(
    cm,
    index=[f"true_{label}" for label in LABELS_5WAY],
    columns=[f"pred_{label}" for label in LABELS_5WAY]
)

print("\nConfusion Matrix - 5-way model")
display(cm_df)

5-way classification accuracy: 0.5573

Classification Report - 5-way model
                   precision    recall  f1-score   support

             None     0.5864    0.5582    0.5720       541
      beoordeling     0.5187    0.6067    0.5592       389
       beslissing     0.7286    0.7391    0.7338        69
 materiele feiten     0.5581    0.5657    0.5619       297
proceshandelingen     0.4969    0.3883    0.4360       206

         accuracy                         0.5573      1502
        macro avg     0.5777    0.5716    0.5726      1502
     weighted avg     0.5575    0.5573    0.5555      1502


Raw Confusion Matrix (5-way model)
[[302 140  13  51  35]
 [ 64 236   5  54  30]
 [ 15   2  51   0   1]
 [ 81  32   1 168  15]
 [ 53  45   0  28  80]]

Confusion Matrix - 5-way model


,pred_None,pred_beoordeling,pred_beslissing,pred_materiele feiten,pred_proceshandelingen
true_None,302,140,13,51,35
true_beoordeling,64,236,5,54,30
true_beslissing,15,2,51,0,1
true_materiele feiten,81,32,1,168,15
true_proceshandelingen,53,45,0,28,80


## OVR Models 

In [5]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import pandas as pd

# ------------------------------------------------------------
# OvR evaluation on Stage 2 legal-only subset
# Excludes gold label == "None"
# ------------------------------------------------------------

LEGAL_LABELS = [
    "beoordeling",
    "beslissing",
    "materiele feiten",
    "proceshandelingen"
]

ovr_settings = {
    "beoordeling": "lr_beoordeling_pred",
    "beslissing": "lr_beslissing_pred",
    "materiele feiten": "lr_materiele_feiten_pred",
    "proceshandelingen": "lr_proceshandelingen_pred",
}

# Only evaluate OvR models on sentences that truly belong to one of the 4 legal classes
ovr_eval_df = lr_output_df[lr_output_df["label"].isin(LEGAL_LABELS)].copy()

print("Total rows in full dataframe:", len(lr_output_df))
print("Rows used for OvR Stage 2 evaluation:", len(ovr_eval_df))
print("Excluded None rows:", len(lr_output_df) - len(ovr_eval_df))


# ------------------------------------------------------------
# Evaluate each OvR model separately
# ------------------------------------------------------------

for target_label, pred_col in ovr_settings.items():
    
    print("\n" + "=" * 80)
    print(f"OvR evaluation for: {target_label}")
    print("=" * 80)
    
    if pred_col not in ovr_eval_df.columns:
        raise ValueError(f"Prediction column not found: {pred_col}")
    
    # Binary gold labels:
    # 1 = current target label
    # 0 = any other legal label
    y_true = (ovr_eval_df["label"] == target_label).astype(int)
    
    # Binary model prediction:
    # 1 = predicted as target label
    # 0 = predicted as not target label
    y_pred = ovr_eval_df[pred_col].astype(int)
    
    acc = accuracy_score(y_true, y_pred)
    print(f"\nAccuracy for {target_label}: {acc:.4f}")
    
    print("\nClassification Report")
    print(
        classification_report(
            y_true,
            y_pred,
            labels=[0, 1],
            target_names=[f"not_{target_label}", target_label],
            digits=4,
            zero_division=0
        )
    )
    
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    
    cm_df = pd.DataFrame(
        cm,
        index=[f"true_not_{target_label}", f"true_{target_label}"],
        columns=[f"pred_not_{target_label}", f"pred_{target_label}"]
    )
    
    print("Confusion Matrix")
    display(cm_df)

Total rows in full dataframe: 1502
Rows used for OvR Stage 2 evaluation: 961
Excluded None rows: 541

OvR evaluation for: beoordeling

Accuracy for beoordeling: 0.7596

Classification Report
                 precision    recall  f1-score   support

not_beoordeling     0.8106    0.7780    0.7939       572
    beoordeling     0.6917    0.7326    0.7116       389

       accuracy                         0.7596       961
      macro avg     0.7512    0.7553    0.7528       961
   weighted avg     0.7625    0.7596    0.7606       961

Confusion Matrix


,pred_not_beoordeling,pred_beoordeling
true_not_beoordeling,445,127
true_beoordeling,104,285



OvR evaluation for: beslissing

Accuracy for beslissing: 0.9771

Classification Report
                precision    recall  f1-score   support

not_beslissing     0.9866    0.9888    0.9877       892
    beslissing     0.8507    0.8261    0.8382        69

      accuracy                         0.9771       961
     macro avg     0.9187    0.9074    0.9130       961
  weighted avg     0.9768    0.9771    0.9770       961

Confusion Matrix


,pred_not_beslissing,pred_beslissing
true_not_beslissing,882,10
true_beslissing,12,57



OvR evaluation for: materiele feiten

Accuracy for materiele feiten: 0.8033

Classification Report
                      precision    recall  f1-score   support

not_materiele feiten     0.8740    0.8358    0.8545       664
    materiele feiten     0.6656    0.7306    0.6966       297

            accuracy                         0.8033       961
           macro avg     0.7698    0.7832    0.7756       961
        weighted avg     0.8096    0.8033    0.8057       961

Confusion Matrix


,pred_not_materiele feiten,pred_materiele feiten
true_not_materiele feiten,555,109
true_materiele feiten,80,217



OvR evaluation for: proceshandelingen

Accuracy for proceshandelingen: 0.8200

Classification Report
                       precision    recall  f1-score   support

not_proceshandelingen     0.8610    0.9192    0.8892       755
    proceshandelingen     0.6065    0.4563    0.5208       206

             accuracy                         0.8200       961
            macro avg     0.7337    0.6878    0.7050       961
         weighted avg     0.8065    0.8200    0.8102       961

Confusion Matrix


,pred_not_proceshandelingen,pred_proceshandelingen
true_not_proceshandelingen,694,61
true_proceshandelingen,112,94


## Stage 1: Binary Gatekeeper

In [6]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import pandas as pd

# ------------------------------------------------------------
# Stage 1 gatekeeper evaluation
# Binary: None vs labelled
# ------------------------------------------------------------

STAGE1_LABELS = [0, 1]
STAGE1_NAMES = ["not_labelled / None", "labelled"]

y_true = lr_output_df["y_labelled"].astype(int)
y_pred = lr_output_df["lr_stage1_pred_idx"].astype(int)

# ------------------------------------------------------------
# Accuracy
# ------------------------------------------------------------

acc = accuracy_score(y_true, y_pred)
print("Stage 1 gatekeeper accuracy:", round(acc, 4))

# ------------------------------------------------------------
# Classification report
# ------------------------------------------------------------

print("\nClassification Report - Stage 1 gatekeeper")
print(
    classification_report(
        y_true,
        y_pred,
        labels=STAGE1_LABELS,
        target_names=STAGE1_NAMES,
        digits=4,
        zero_division=0
    )
)

# ------------------------------------------------------------
# Confusion matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=STAGE1_LABELS
)

cm_df = pd.DataFrame(
    cm,
    index=["true_not_labelled_None", "true_labelled"],
    columns=["pred_not_labelled_None", "pred_labelled"]
)

print("\nConfusion Matrix - Stage 1 gatekeeper")
display(cm_df)

Stage 1 gatekeeper accuracy: 0.6851

Classification Report - Stage 1 gatekeeper
                     precision    recall  f1-score   support

not_labelled / None     0.5528    0.6580    0.6008       541
           labelled     0.7844    0.7003    0.7400       961

           accuracy                         0.6851      1502
          macro avg     0.6686    0.6792    0.6704      1502
       weighted avg     0.7010    0.6851    0.6899      1502


Confusion Matrix - Stage 1 gatekeeper


,pred_not_labelled_None,pred_labelled
true_not_labelled_None,356,185
true_labelled,288,673


## Stage 2: 4 way (Without None)

In [7]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import pandas as pd

# ------------------------------------------------------------
# Stage 2 4-way evaluation
# Only evaluate on gold labelled / legal-role sentences
# ------------------------------------------------------------

STAGE2_LABELS = [
    "beoordeling",
    "beslissing",
    "materiele feiten",
    "proceshandelingen"
]

# Keep only sentences that truly belong to one of the 4 legal labels
stage2_eval_df = lr_output_df[
    lr_output_df["label"].isin(STAGE2_LABELS)
].copy()

print("Total rows in full dataframe:", len(lr_output_df))
print("Rows used for Stage 2 4-way evaluation:", len(stage2_eval_df))
print("Excluded None rows:", len(lr_output_df) - len(stage2_eval_df))

# ------------------------------------------------------------
# Gold and predicted labels
# ------------------------------------------------------------

y_true = stage2_eval_df["label"]
y_pred = stage2_eval_df["lr_stage2_pred_label"]

# ------------------------------------------------------------
# Accuracy
# ------------------------------------------------------------

acc = accuracy_score(y_true, y_pred)
print("\nStage 2 4-way accuracy:", round(acc, 4))

# ------------------------------------------------------------
# Classification report
# ------------------------------------------------------------

print("\nClassification Report - Stage 2 4-way model")
print(
    classification_report(
        y_true,
        y_pred,
        labels=STAGE2_LABELS,
        target_names=STAGE2_LABELS,
        digits=4,
        zero_division=0
    )
)

# ------------------------------------------------------------
# Confusion matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=STAGE2_LABELS
)

cm_df = pd.DataFrame(
    cm,
    index=[f"true_{label}" for label in STAGE2_LABELS],
    columns=[f"pred_{label}" for label in STAGE2_LABELS]
)

print("\nConfusion Matrix - Stage 2 4-way model")
display(cm_df)

Total rows in full dataframe: 1502
Rows used for Stage 2 4-way evaluation: 961
Excluded None rows: 541

Stage 2 4-way accuracy: 0.6795

Classification Report - Stage 2 4-way model
                   precision    recall  f1-score   support

      beoordeling     0.6878    0.7249    0.7059       389
       beslissing     0.8788    0.8406    0.8593        69
 materiele feiten     0.6636    0.7306    0.6955       297
proceshandelingen     0.6076    0.4660    0.5275       206

         accuracy                         0.6795       961
        macro avg     0.7094    0.6905    0.6970       961
     weighted avg     0.6768    0.6795    0.6754       961


Confusion Matrix - Stage 2 4-way model


,pred_beoordeling,pred_beslissing,pred_materiele feiten,pred_proceshandelingen
true_beoordeling,282,5,65,37
true_beslissing,5,58,4,2
true_materiele feiten,56,1,217,23
true_proceshandelingen,67,2,41,96


## Bayes Update

In [8]:
import numpy as np
import pandas as pd
import re

# ------------------------------------------------------------
# 0. Setup
# ------------------------------------------------------------

# Use the LR output dataframe you created earlier
lr_bayes_df = lr_output_df.copy()

# Clean labels
train_df["label"] = train_df["label"].fillna("None").astype(str).str.strip()
lr_bayes_df["label"] = lr_bayes_df["label"].fillna("None").astype(str).str.strip()

LAM = 3.0
ALPHA = 1.0

ALL_LABELS = [
    "None",
    "beoordeling",
    "beslissing",
    "materiele feiten",
    "proceshandelingen"
]

LEGAL_LABELS = [
    "beoordeling",
    "beslissing",
    "materiele feiten",
    "proceshandelingen"
]


# ------------------------------------------------------------
# 1. Helper functions
# ------------------------------------------------------------

def clean_col_name(x):
    x = str(x).strip().lower()
    x = re.sub(r"\s+", "_", x)
    x = re.sub(r"[^a-zA-Z0-9_]+", "_", x)
    x = re.sub(r"_+", "_", x).strip("_")
    return x


def make_header_prior(df_train, label_col, labels, alpha=1.0):
    """
    Estimate P(label | hdr_group) from training data only.
    Laplace smoothing is applied.
    """
    counts = (
        df_train
        .groupby(["hdr_group", label_col])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=labels, fill_value=0)
    )

    probs = counts + alpha
    probs = probs.div(probs.sum(axis=1), axis=0)

    return probs


def make_global_prior(df_train, label_col, labels, alpha=1.0):
    """
    Fallback prior used when a header group is unseen.
    """
    counts = df_train[label_col].value_counts().reindex(labels, fill_value=0) + alpha
    prior = counts / counts.sum()
    return prior.values


def get_meta_prior_df(df_apply, header_prior_train, global_prior, labels):
    """
    Create one metadata-prior row per sentence in the dataframe being predicted.
    """
    def get_prior_for_row(hdr_group):
        if pd.notna(hdr_group) and hdr_group in header_prior_train.index:
            return header_prior_train.loc[hdr_group].values
        return global_prior

    meta_prior_mat = np.vstack(
        df_apply["hdr_group"].apply(get_prior_for_row)
    )

    return pd.DataFrame(
        meta_prior_mat,
        columns=labels,
        index=df_apply.index
    )


def combine_log_scores(text_probs_df, meta_prior_df, lam=1.0, eps=1e-12):
    """
    Bayesian-style log fusion:
    log score = log P(label | text) + lambda * log P(label | header)
    """
    text = np.clip(text_probs_df.values, eps, 1.0)
    meta = np.clip(meta_prior_df.values, eps, 1.0)

    log_score = np.log(text) + lam * np.log(meta)

    # numerical stability
    log_score = log_score - log_score.max(axis=1, keepdims=True)

    score = np.exp(log_score)
    score = score / score.sum(axis=1, keepdims=True)

    return pd.DataFrame(
        score,
        columns=text_probs_df.columns,
        index=text_probs_df.index
    )


def add_bayes_prob_columns(df, prefix, bayes_probs, labels):
    """
    Add Bayesian-updated probability columns and prediction label.
    """
    for lab in labels:
        clean_lab = clean_col_name(lab)
        df[f"{prefix}_p_{clean_lab}"] = bayes_probs[lab]

    df[f"{prefix}_pred_label"] = bayes_probs.idxmax(axis=1)

    return df

In [13]:
# ------------------------------------------------------------
# LR 5-way Bayesian update
# ------------------------------------------------------------

lr_5way_prob_cols = {
    "None": "lr_5way_p_none",
    "beoordeling": "lr_5way_p_beoordeling",
    "beslissing": "lr_5way_p_beslissing",
    "materiele feiten": "lr_5way_p_materiele_feiten",
    "proceshandelingen": "lr_5way_p_proceshandelingen"
}

lr_5way_text_probs = lr_bayes_df[
    [lr_5way_prob_cols[label] for label in ALL_LABELS]
].copy()

lr_5way_text_probs.columns = ALL_LABELS

header_prior_5way = make_header_prior(
    df_train=train_df,
    label_col="label",
    labels=ALL_LABELS,
    alpha=ALPHA
)

global_prior_5way = make_global_prior(
    df_train=train_df,
    label_col="label",
    labels=ALL_LABELS,
    alpha=ALPHA
)

meta_prior_5way = get_meta_prior_df(
    df_apply=lr_bayes_df,
    header_prior_train=header_prior_5way,
    global_prior=global_prior_5way,
    labels=ALL_LABELS
)

lr_bayes_5way_probs = combine_log_scores(
    text_probs_df=lr_5way_text_probs,
    meta_prior_df=meta_prior_5way,
    lam=LAM
)

lr_bayes_df = add_bayes_prob_columns(
    df=lr_bayes_df,
    prefix="lr_bayes_5way",
    bayes_probs=lr_bayes_5way_probs,
    labels=ALL_LABELS
)

print("Done: LR 5-way Bayesian update")
lr_bayes_df[["label", "lr_5way_pred_label", "lr_bayes_5way_pred_label"]].head()

# ------------------------------------------------------------
# LR + Bayes 5-way evaluation
# ------------------------------------------------------------

y_true = lr_bayes_df["label"]
y_pred = lr_bayes_df["lr_bayes_5way_pred_label"]

acc = accuracy_score(y_true, y_pred)
print("LR + Bayes 5-way accuracy:", round(acc, 4))

print("\nClassification Report - LR + Bayes 5-way")
print(
    classification_report(
        y_true,
        y_pred,
        labels=ALL_LABELS,
        target_names=ALL_LABELS,
        digits=4,
        zero_division=0
    )
)

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=ALL_LABELS
)

cm_df = pd.DataFrame(
    cm,
    index=[f"true_{label}" for label in ALL_LABELS],
    columns=[f"pred_{label}" for label in ALL_LABELS]
)

print("\nConfusion Matrix - LR + Bayes 5-way")
display(cm_df)

Done: LR 5-way Bayesian update
LR + Bayes 5-way accuracy: 0.5033

Classification Report - LR + Bayes 5-way
                   precision    recall  f1-score   support

             None     0.4614    0.6192    0.5288       541
      beoordeling     0.4981    0.6812    0.5755       389
       beslissing     0.8696    0.2899    0.4348        69
 materiele feiten     0.8667    0.0438    0.0833       297
proceshandelingen     0.5971    0.5971    0.5971       206

         accuracy                         0.5033      1502
        macro avg     0.6586    0.4462    0.4439      1502
     weighted avg     0.5884    0.5033    0.4578      1502


Confusion Matrix - LR + Bayes 5-way


,pred_None,pred_beoordeling,pred_beslissing,pred_materiele feiten,pred_proceshandelingen
true_None,335,155,2,2,47
true_beoordeling,89,265,1,0,34
true_beslissing,47,0,20,0,2
true_materiele feiten,227,57,0,13,0
true_proceshandelingen,28,55,0,0,123


In [14]:
# ------------------------------------------------------------
# LR Stage 1 Bayesian update
# ------------------------------------------------------------

stage1_train = train_df.copy()
stage1_train["y_stage1"] = np.where(
    stage1_train["label"] == "None",
    0,
    1
)

STAGE1_LABELS = [0, 1]

lr_stage1_text_probs = lr_bayes_df[
    ["lr_stage1_p0", "lr_stage1_p1"]
].copy()

lr_stage1_text_probs.columns = STAGE1_LABELS

header_prior_stage1 = make_header_prior(
    df_train=stage1_train,
    label_col="y_stage1",
    labels=STAGE1_LABELS,
    alpha=ALPHA
)

global_prior_stage1 = make_global_prior(
    df_train=stage1_train,
    label_col="y_stage1",
    labels=STAGE1_LABELS,
    alpha=ALPHA
)

meta_prior_stage1 = get_meta_prior_df(
    df_apply=lr_bayes_df,
    header_prior_train=header_prior_stage1,
    global_prior=global_prior_stage1,
    labels=STAGE1_LABELS
)

lr_bayes_stage1_probs = combine_log_scores(
    text_probs_df=lr_stage1_text_probs,
    meta_prior_df=meta_prior_stage1,
    lam=LAM
)

lr_bayes_df["lr_bayes_stage1_p0"] = lr_bayes_stage1_probs[0]
lr_bayes_df["lr_bayes_stage1_p1"] = lr_bayes_stage1_probs[1]

lr_bayes_df["lr_bayes_stage1_pred_idx"] = lr_bayes_stage1_probs.idxmax(axis=1)

lr_bayes_df["lr_bayes_stage1_pred_label"] = np.where(
    lr_bayes_df["lr_bayes_stage1_pred_idx"] == 0,
    "not_labelled",
    "labelled"
)

print("Done: LR Stage 1 Bayesian update")
lr_bayes_df[
    ["y_labelled", "lr_stage1_pred_idx", "lr_bayes_stage1_pred_idx", 
     "lr_stage1_p1", "lr_bayes_stage1_p1"]
].head()


# ------------------------------------------------------------
# LR + Bayes Stage 1 gatekeeper evaluation
# Binary: None vs labelled
# ------------------------------------------------------------

STAGE1_LABELS = [0, 1]
STAGE1_NAMES = ["not_labelled / None", "labelled"]

y_true = lr_bayes_df["y_labelled"].astype(int)
y_pred = lr_bayes_df["lr_bayes_stage1_pred_idx"].astype(int)

acc = accuracy_score(y_true, y_pred)
print("LR + Bayes Stage 1 accuracy:", round(acc, 4))

print("\nClassification Report - LR + Bayes Stage 1")
print(
    classification_report(
        y_true,
        y_pred,
        labels=STAGE1_LABELS,
        target_names=STAGE1_NAMES,
        digits=4,
        zero_division=0
    )
)

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=STAGE1_LABELS
)

cm_df = pd.DataFrame(
    cm,
    index=["true_not_labelled_None", "true_labelled"],
    columns=["pred_not_labelled_None", "pred_labelled"]
)

print("\nConfusion Matrix - LR + Bayes Stage 1")
display(cm_df)

Done: LR Stage 1 Bayesian update
LR + Bayes Stage 1 accuracy: 0.6711

Classification Report - LR + Bayes Stage 1
                     precision    recall  f1-score   support

not_labelled / None     0.5630    0.3882    0.4595       541
           labelled     0.7068    0.8304    0.7636       961

           accuracy                         0.6711      1502
          macro avg     0.6349    0.6093    0.6116      1502
       weighted avg     0.6550    0.6711    0.6541      1502


Confusion Matrix - LR + Bayes Stage 1


,pred_not_labelled_None,pred_labelled
true_not_labelled_None,210,331
true_labelled,163,798


In [15]:
# ------------------------------------------------------------
# LR Stage 2 4-way Bayesian update
# ------------------------------------------------------------

stage2_train = train_df[
    train_df["label"].isin(LEGAL_LABELS)
].copy()

lr_stage2_prob_cols = {
    "beoordeling": "lr_stage2_p_beoordeling",
    "beslissing": "lr_stage2_p_beslissing",
    "materiele feiten": "lr_stage2_p_materiele_feiten",
    "proceshandelingen": "lr_stage2_p_proceshandelingen"
}

lr_stage2_text_probs = lr_bayes_df[
    [lr_stage2_prob_cols[label] for label in LEGAL_LABELS]
].copy()

lr_stage2_text_probs.columns = LEGAL_LABELS

header_prior_stage2 = make_header_prior(
    df_train=stage2_train,
    label_col="label",
    labels=LEGAL_LABELS,
    alpha=ALPHA
)

global_prior_stage2 = make_global_prior(
    df_train=stage2_train,
    label_col="label",
    labels=LEGAL_LABELS,
    alpha=ALPHA
)

meta_prior_stage2 = get_meta_prior_df(
    df_apply=lr_bayes_df,
    header_prior_train=header_prior_stage2,
    global_prior=global_prior_stage2,
    labels=LEGAL_LABELS
)

lr_bayes_stage2_probs = combine_log_scores(
    text_probs_df=lr_stage2_text_probs,
    meta_prior_df=meta_prior_stage2,
    lam=LAM
)

lr_bayes_df = add_bayes_prob_columns(
    df=lr_bayes_df,
    prefix="lr_bayes_stage2",
    bayes_probs=lr_bayes_stage2_probs,
    labels=LEGAL_LABELS
)

print("Done: LR Stage 2 Bayesian update")
lr_bayes_df[["label", "lr_stage2_pred_label", "lr_bayes_stage2_pred_label"]].head()

# ------------------------------------------------------------
# LR + Bayes Stage 2 4-way evaluation
# Gold legal-labelled sentences only
# ------------------------------------------------------------

LEGAL_LABELS = [
    "beoordeling",
    "beslissing",
    "materiele feiten",
    "proceshandelingen"
]

stage2_eval_df = lr_bayes_df[
    lr_bayes_df["label"].isin(LEGAL_LABELS)
].copy()

print("Total rows:", len(lr_bayes_df))
print("Rows used for Stage 2 evaluation:", len(stage2_eval_df))
print("Excluded None rows:", len(lr_bayes_df) - len(stage2_eval_df))

y_true = stage2_eval_df["label"]
y_pred = stage2_eval_df["lr_bayes_stage2_pred_label"]

acc = accuracy_score(y_true, y_pred)
print("\nLR + Bayes Stage 2 accuracy:", round(acc, 4))

print("\nClassification Report - LR + Bayes Stage 2")
print(
    classification_report(
        y_true,
        y_pred,
        labels=LEGAL_LABELS,
        target_names=LEGAL_LABELS,
        digits=4,
        zero_division=0
    )
)

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=LEGAL_LABELS
)

cm_df = pd.DataFrame(
    cm,
    index=[f"true_{label}" for label in LEGAL_LABELS],
    columns=[f"pred_{label}" for label in LEGAL_LABELS]
)

print("\nConfusion Matrix - LR + Bayes Stage 2")
display(cm_df)

Done: LR Stage 2 Bayesian update
Total rows: 1502
Rows used for Stage 2 evaluation: 961
Excluded None rows: 541

LR + Bayes Stage 2 accuracy: 0.7086

Classification Report - LR + Bayes Stage 2
                   precision    recall  f1-score   support

      beoordeling     0.6080    0.8612    0.7128       389
       beslissing     0.9394    0.8986    0.9185        69
 materiele feiten     0.9086    0.5354    0.6737       297
proceshandelingen     0.7396    0.6068    0.6667       206

         accuracy                         0.7086       961
        macro avg     0.7989    0.7255    0.7429       961
     weighted avg     0.7529    0.7086    0.7056       961


Confusion Matrix - LR + Bayes Stage 2


,pred_beoordeling,pred_beslissing,pred_materiele feiten,pred_proceshandelingen
true_beoordeling,335,1,12,41
true_beslissing,3,62,1,3
true_materiele feiten,135,3,159,0
true_proceshandelingen,78,0,3,125


In [16]:
# ------------------------------------------------------------
# LR OvR Bayesian update
# ------------------------------------------------------------

ovr_train_base = train_df[
    train_df["label"].isin(LEGAL_LABELS)
].copy()

for target_label in LEGAL_LABELS:

    clean_label = clean_col_name(target_label)

    print(f"Running LR OvR Bayesian update for: {target_label}")

    ovr_train = ovr_train_base.copy()
    ovr_train[f"y_{clean_label}"] = np.where(
        ovr_train["label"] == target_label,
        1,
        0
    )

    binary_labels = [0, 1]

    text_probs = lr_bayes_df[
        [f"lr_{clean_label}_p_not", f"lr_{clean_label}_p_pos"]
    ].copy()

    text_probs.columns = binary_labels

    header_prior_ovr = make_header_prior(
        df_train=ovr_train,
        label_col=f"y_{clean_label}",
        labels=binary_labels,
        alpha=ALPHA
    )

    global_prior_ovr = make_global_prior(
        df_train=ovr_train,
        label_col=f"y_{clean_label}",
        labels=binary_labels,
        alpha=ALPHA
    )

    meta_prior_ovr = get_meta_prior_df(
        df_apply=lr_bayes_df,
        header_prior_train=header_prior_ovr,
        global_prior=global_prior_ovr,
        labels=binary_labels
    )

    bayes_probs = combine_log_scores(
        text_probs_df=text_probs,
        meta_prior_df=meta_prior_ovr,
        lam=LAM
    )

    lr_bayes_df[f"lr_bayes_{clean_label}_p_not"] = bayes_probs[0]
    lr_bayes_df[f"lr_bayes_{clean_label}_p_pos"] = bayes_probs[1]
    lr_bayes_df[f"lr_bayes_{clean_label}_pred"] = bayes_probs.idxmax(axis=1)

print("Done: LR OvR Bayesian update")


# ------------------------------------------------------------
# LR + Bayes OvR evaluation
# Each binary model evaluated on legal-labelled subset
# ------------------------------------------------------------

ovr_eval_df = lr_bayes_df[
    lr_bayes_df["label"].isin(LEGAL_LABELS)
].copy()

ovr_settings = {
    "beoordeling": "lr_bayes_beoordeling_pred",
    "beslissing": "lr_bayes_beslissing_pred",
    "materiele feiten": "lr_bayes_materiele_feiten_pred",
    "proceshandelingen": "lr_bayes_proceshandelingen_pred",
}

print("Total rows:", len(lr_bayes_df))
print("Rows used for OvR evaluation:", len(ovr_eval_df))
print("Excluded None rows:", len(lr_bayes_df) - len(ovr_eval_df))

for target_label, pred_col in ovr_settings.items():

    print("\n" + "=" * 80)
    print(f"LR + Bayes OvR evaluation for: {target_label}")
    print("=" * 80)

    y_true = (ovr_eval_df["label"] == target_label).astype(int)
    y_pred = ovr_eval_df[pred_col].astype(int)

    acc = accuracy_score(y_true, y_pred)
    print(f"\nAccuracy for {target_label}: {acc:.4f}")

    print("\nClassification Report")
    print(
        classification_report(
            y_true,
            y_pred,
            labels=[0, 1],
            target_names=[f"not_{target_label}", target_label],
            digits=4,
            zero_division=0
        )
    )

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    cm_df = pd.DataFrame(
        cm,
        index=[f"true_not_{target_label}", f"true_{target_label}"],
        columns=[f"pred_not_{target_label}", f"pred_{target_label}"]
    )

    print("Confusion Matrix")
    display(cm_df)

Running LR OvR Bayesian update for: beoordeling
Running LR OvR Bayesian update for: beslissing
Running LR OvR Bayesian update for: materiele feiten
Running LR OvR Bayesian update for: proceshandelingen
Done: LR OvR Bayesian update
Total rows: 1502
Rows used for OvR evaluation: 961
Excluded None rows: 541

LR + Bayes OvR evaluation for: beoordeling

Accuracy for beoordeling: 0.7971

Classification Report
                 precision    recall  f1-score   support

not_beoordeling     0.7941    0.8899    0.8392       572
    beoordeling     0.8031    0.6607    0.7250       389

       accuracy                         0.7971       961
      macro avg     0.7986    0.7753    0.7821       961
   weighted avg     0.7977    0.7971    0.7930       961

Confusion Matrix


,pred_not_beoordeling,pred_beoordeling
true_not_beoordeling,509,63
true_beoordeling,132,257



LR + Bayes OvR evaluation for: beslissing

Accuracy for beslissing: 0.9729

Classification Report
                precision    recall  f1-score   support

not_beslissing     0.9727    0.9989    0.9856       892
    beslissing     0.9778    0.6377    0.7719        69

      accuracy                         0.9729       961
     macro avg     0.9752    0.8183    0.8788       961
  weighted avg     0.9731    0.9729    0.9703       961

Confusion Matrix


,pred_not_beslissing,pred_beslissing
true_not_beslissing,891,1
true_beslissing,25,44



LR + Bayes OvR evaluation for: materiele feiten

Accuracy for materiele feiten: 0.8398

Classification Report
                      precision    recall  f1-score   support

not_materiele feiten     0.8164    0.9910    0.8952       664
    materiele feiten     0.9613    0.5017    0.6593       297

            accuracy                         0.8398       961
           macro avg     0.8888    0.7463    0.7773       961
        weighted avg     0.8612    0.8398    0.8223       961

Confusion Matrix


,pred_not_materiele feiten,pred_materiele feiten
true_not_materiele feiten,658,6
true_materiele feiten,148,149



LR + Bayes OvR evaluation for: proceshandelingen

Accuracy for proceshandelingen: 0.8845

Classification Report
                       precision    recall  f1-score   support

not_proceshandelingen     0.8966    0.9642    0.9292       755
    proceshandelingen     0.8188    0.5922    0.6873       206

             accuracy                         0.8845       961
            macro avg     0.8577    0.7782    0.8082       961
         weighted avg     0.8799    0.8845    0.8773       961

Confusion Matrix


,pred_not_proceshandelingen,pred_proceshandelingen
true_not_proceshandelingen,728,27
true_proceshandelingen,84,122


Pipeline evaluations

In [25]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

ALL_LABELS = [
    "None",
    "beoordeling",
    "beslissing",
    "materiele feiten",
    "proceshandelingen"
]

LEGAL_LABELS = [
    "beoordeling",
    "beslissing",
    "materiele feiten",
    "proceshandelingen"
]


def make_pipeline_A_probabilistic(
    df,
    stage1_p_none_col,
    stage1_p_labelled_col,
    stage2_prob_cols,
    prefix
):
    """
    Pipeline A probabilistic combination.

    Final 5-way probabilities:
        P(None) = P(Stage 1 = None)
        P(legal label) = P(Stage 1 = labelled) * P(Stage 2 = legal label | labelled)
    """

    df = df.copy()

    p_none = df[stage1_p_none_col].astype(float).values
    p_labelled = df[stage1_p_labelled_col].astype(float).values

    df[f"{prefix}_p_none"] = p_none

    for label in LEGAL_LABELS:
        clean_label = label.replace(" ", "_")
        df[f"{prefix}_p_{clean_label}"] = (
            p_labelled * df[stage2_prob_cols[label]].astype(float).values
        )

    final_prob_cols = [
        f"{prefix}_p_none",
        f"{prefix}_p_beoordeling",
        f"{prefix}_p_beslissing",
        f"{prefix}_p_materiele_feiten",
        f"{prefix}_p_proceshandelingen"
    ]

    probs_for_pred = df[final_prob_cols].copy()
    probs_for_pred.columns = ALL_LABELS

    df[f"{prefix}_pred"] = probs_for_pred.idxmax(axis=1)

    return df


def evaluate_5way_prediction(df, pred_col, title):
    y_true = df["label"].fillna("None").astype(str).str.strip()
    y_pred = df[pred_col].fillna("None").astype(str).str.strip()

    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)

    acc = accuracy_score(y_true, y_pred)
    print(f"\nAccuracy: {acc:.4f}")

    print("\nClassification Report")
    print(
        classification_report(
            y_true,
            y_pred,
            labels=ALL_LABELS,
            target_names=ALL_LABELS,
            digits=4,
            zero_division=0
        )
    )

    cm = confusion_matrix(y_true, y_pred, labels=ALL_LABELS)

    cm_df = pd.DataFrame(
        cm,
        index=[f"true_{label}" for label in ALL_LABELS],
        columns=[f"pred_{label}" for label in ALL_LABELS]
    )

    print("\nConfusion Matrix")
    display(cm_df)

In [26]:
# ------------------------------------------------------------
# Logistic Regression Pipeline A: Stage 1 + Stage 2
# Before Bayesian updating
# ------------------------------------------------------------

lr_stage2_prob_cols = {
    "beoordeling": "lr_stage2_p_beoordeling",
    "beslissing": "lr_stage2_p_beslissing",
    "materiele feiten": "lr_stage2_p_materiele_feiten",
    "proceshandelingen": "lr_stage2_p_proceshandelingen"
}

lr_output_df = make_pipeline_A_probabilistic(
    df=lr_output_df,
    stage1_p_none_col="lr_stage1_p0",
    stage1_p_labelled_col="lr_stage1_p1",
    stage2_prob_cols=lr_stage2_prob_cols,
    prefix="lr_pipeline_A_prob"
)

evaluate_5way_prediction(
    df=lr_output_df,
    pred_col="lr_pipeline_A_prob_pred",
    title="Logistic Regression Pipeline A: Probabilistic Stage 1 + Stage 2"
)


Logistic Regression Pipeline A: Probabilistic Stage 1 + Stage 2

Accuracy: 0.5233

Classification Report
                   precision    recall  f1-score   support

             None     0.4783    0.8558    0.6137       541
      beoordeling     0.6212    0.4216    0.5023       389
       beslissing     0.8571    0.6087    0.7119        69
 materiele feiten     0.5267    0.2323    0.3224       297
proceshandelingen     0.5333    0.2330    0.3243       206

         accuracy                         0.5233      1502
        macro avg     0.6033    0.4703    0.4949      1502
     weighted avg     0.5498    0.5233    0.4921      1502


Confusion Matrix


,pred_None,pred_beoordeling,pred_beslissing,pred_materiele feiten,pred_proceshandelingen
true_None,463,49,2,12,15
true_beoordeling,167,164,5,36,17
true_beslissing,26,1,42,0,0
true_materiele feiten,200,18,0,69,10
true_proceshandelingen,112,32,0,14,48


In [27]:
# ------------------------------------------------------------
# Logistic Regression Pipeline A: Stage 1 + Stage 2
# After Bayesian updating
# ------------------------------------------------------------

lr_bayes_stage2_prob_cols = {
    "beoordeling": "lr_bayes_stage2_p_beoordeling",
    "beslissing": "lr_bayes_stage2_p_beslissing",
    "materiele feiten": "lr_bayes_stage2_p_materiele_feiten",
    "proceshandelingen": "lr_bayes_stage2_p_proceshandelingen"
}

lr_bayes_df = make_pipeline_A_probabilistic(
    df=lr_bayes_df,
    stage1_p_none_col="lr_bayes_stage1_p0",
    stage1_p_labelled_col="lr_bayes_stage1_p1",
    stage2_prob_cols=lr_bayes_stage2_prob_cols,
    prefix="lr_bayes_pipeline_A_prob"
)

evaluate_5way_prediction(
    df=lr_bayes_df,
    pred_col="lr_bayes_pipeline_A_prob_pred",
    title="Logistic Regression + Bayes Pipeline A: Probabilistic Stage 1 + Stage 2"
)


Logistic Regression + Bayes Pipeline A: Probabilistic Stage 1 + Stage 2

Accuracy: 0.4880

Classification Report
                   precision    recall  f1-score   support

             None     0.5427    0.3993    0.4601       541
      beoordeling     0.4203    0.8201    0.5557       389
       beslissing     0.7869    0.6957    0.7385        69
 materiele feiten     0.5435    0.0842    0.1458       297
proceshandelingen     0.5252    0.6068    0.5631       206

         accuracy                         0.4880      1502
        macro avg     0.5637    0.5212    0.4926      1502
     weighted avg     0.5200    0.4880    0.4496      1502


Confusion Matrix


,pred_None,pred_beoordeling,pred_beslissing,pred_materiele feiten,pred_proceshandelingen
true_None,216,236,9,11,69
true_beoordeling,20,319,1,8,41
true_beslissing,18,0,48,0,3
true_materiele feiten,141,128,3,25,0
true_proceshandelingen,3,76,0,2,125


In [22]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

ALL_LABELS = [
    "None",
    "beoordeling",
    "beslissing",
    "materiele feiten",
    "proceshandelingen"
]

LEGAL_LABELS = [
    "beoordeling",
    "beslissing",
    "materiele feiten",
    "proceshandelingen"
]


def make_pipeline_B_probabilistic(
    df,
    stage1_p_none_col,
    stage1_p_labelled_col,
    ovr_pos_cols,
    legal_labels,
    prefix,
    eps=1e-12
):
    """
    Pipeline B probabilistic combination.

    Step 1:
        Use Stage 1 probabilities:
        P(None)
        P(labelled)

    Step 2:
        Normalize the four OvR positive probabilities across legal labels:
        q_k = p_ovr_k / sum(p_ovr_all)

    Step 3:
        Construct final 5-way probabilities:
        P(None) = stage1_p_none
        P(k)    = stage1_p_labelled * q_k

    This creates a proper 5-way probability distribution.
    """

    df = df.copy()

    # Stage 1 probabilities
    p_none = df[stage1_p_none_col].astype(float).values
    p_labelled = df[stage1_p_labelled_col].astype(float).values

    # OvR positive probabilities
    ovr_scores = df[ovr_pos_cols].astype(float).values
    ovr_scores = np.clip(ovr_scores, eps, 1.0)

    # Normalize OvR probabilities across the four legal labels
    ovr_norm = ovr_scores / ovr_scores.sum(axis=1, keepdims=True)

    # Final 5-way probabilities
    df[f"{prefix}_p_none"] = p_none

    for i, label in enumerate(legal_labels):
        clean_label = label.replace(" ", "_")
        df[f"{prefix}_p_{clean_label}"] = p_labelled * ovr_norm[:, i]

    final_prob_cols = [
        f"{prefix}_p_none",
        f"{prefix}_p_beoordeling",
        f"{prefix}_p_beslissing",
        f"{prefix}_p_materiele_feiten",
        f"{prefix}_p_proceshandelingen"
    ]

    # Rename temporarily so idxmax gives clean labels
    probs_for_pred = df[final_prob_cols].copy()
    probs_for_pred.columns = ALL_LABELS

    df[f"{prefix}_pred"] = probs_for_pred.idxmax(axis=1)

    return df


def evaluate_5way_prediction(df, pred_col, title):
    y_true = df["label"].fillna("None").astype(str).str.strip()
    y_pred = df[pred_col].fillna("None").astype(str).str.strip()

    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)

    acc = accuracy_score(y_true, y_pred)
    print(f"\nAccuracy: {acc:.4f}")

    print("\nClassification Report")
    print(
        classification_report(
            y_true,
            y_pred,
            labels=ALL_LABELS,
            target_names=ALL_LABELS,
            digits=4,
            zero_division=0
        )
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=ALL_LABELS
    )

    cm_df = pd.DataFrame(
        cm,
        index=[f"true_{label}" for label in ALL_LABELS],
        columns=[f"pred_{label}" for label in ALL_LABELS]
    )

    print("\nConfusion Matrix")
    display(cm_df)

In [23]:
lr_ovr_pos_cols = [
    "lr_beoordeling_p_pos",
    "lr_beslissing_p_pos",
    "lr_materiele_feiten_p_pos",
    "lr_proceshandelingen_p_pos"
]

lr_output_df = make_pipeline_B_probabilistic(
    df=lr_output_df,
    stage1_p_none_col="lr_stage1_p0",
    stage1_p_labelled_col="lr_stage1_p1",
    ovr_pos_cols=lr_ovr_pos_cols,
    legal_labels=LEGAL_LABELS,
    prefix="lr_pipeline_B"
)

evaluate_5way_prediction(
    df=lr_output_df,
    pred_col="lr_pipeline_B_pred",
    title="Logistic Regression Pipeline B: Stage 1 probability + normalized OvR probabilities"
)


Logistic Regression Pipeline B: Stage 1 probability + normalized OvR probabilities

Accuracy: 0.5120

Classification Report
                   precision    recall  f1-score   support

             None     0.4659    0.8706    0.6070       541
      beoordeling     0.6202    0.4113    0.4946       389
       beslissing     0.8537    0.5072    0.6364        69
 materiele feiten     0.5263    0.2020    0.2920       297
proceshandelingen     0.5513    0.2087    0.3028       206

         accuracy                         0.5120      1502
        macro avg     0.6035    0.4400    0.4665      1502
     weighted avg     0.5473    0.5120    0.4752      1502


Confusion Matrix


,pred_None,pred_beoordeling,pred_beslissing,pred_materiele feiten,pred_proceshandelingen
true_None,471,46,1,11,12
true_beoordeling,180,160,5,29,15
true_beslissing,32,2,35,0,0
true_materiele feiten,210,19,0,60,8
true_proceshandelingen,118,31,0,14,43


In [24]:
lr_bayes_ovr_pos_cols = [
    "lr_bayes_beoordeling_p_pos",
    "lr_bayes_beslissing_p_pos",
    "lr_bayes_materiele_feiten_p_pos",
    "lr_bayes_proceshandelingen_p_pos"
]

lr_bayes_df = make_pipeline_B_probabilistic(
    df=lr_bayes_df,
    stage1_p_none_col="lr_bayes_stage1_p0",
    stage1_p_labelled_col="lr_bayes_stage1_p1",
    ovr_pos_cols=lr_bayes_ovr_pos_cols,
    legal_labels=LEGAL_LABELS,
    prefix="lr_bayes_pipeline_B"
)

evaluate_5way_prediction(
    df=lr_bayes_df,
    pred_col="lr_bayes_pipeline_B_pred",
    title="Logistic Regression + Bayes Pipeline B: Stage 1 probability + normalized OvR probabilities"
)


Logistic Regression + Bayes Pipeline B: Stage 1 probability + normalized OvR probabilities

Accuracy: 0.4880

Classification Report
                   precision    recall  f1-score   support

             None     0.5514    0.4067    0.4681       541
      beoordeling     0.4168    0.8175    0.5521       389
       beslissing     0.8704    0.6812    0.7642        69
 materiele feiten     0.5349    0.0774    0.1353       297
proceshandelingen     0.5144    0.6068    0.5568       206

         accuracy                         0.4880      1502
        macro avg     0.5776    0.5179    0.4953      1502
     weighted avg     0.5228    0.4880    0.4498      1502


Confusion Matrix


,pred_None,pred_beoordeling,pred_beslissing,pred_materiele feiten,pred_proceshandelingen
true_None,220,235,3,11,72
true_beoordeling,20,318,1,7,43
true_beslissing,17,2,47,0,3
true_materiele feiten,139,132,3,23,0
true_proceshandelingen,3,76,0,2,125
